# DeltaBind-GNN — End-to-End Colab Pipeline

Clone repo -> download public FEP benchmark data -> parse -> featurize -> train -> evaluate against FEP+ baseline.

**Runtime:** Runtime > Change runtime type > GPU (T4 is fine for this scale of data).

## 1. Setup: clone repo and install dependencies

In [ ]:
# Replace with your own fork/repo URL once pushed to GitHub
REPO_URL = "https://github.com/<your-username>/deltabind-gnn.git"

!git clone $REPO_URL
%cd deltabind-gnn

In [ ]:
!pip install -q torch-geometric rdkit biopython
!pip install -q -r requirements.txt

## 2. Download the public benchmark data (no registration needed)

- Schrodinger public FEP+ benchmark: `github.com/schrodinger/public_binding_free_energy_benchmark`
- Merck/Schindler FEP benchmark: `github.com/MCompChem/fep-benchmark`

In [ ]:
!bash data/download_data.sh

In [ ]:
# Sanity check what actually got cloned before assuming the parser will work
!find data/raw -maxdepth 3 -type d | head -50

## 3. Parse raw benchmark data into a unified edge table

This step is data-source-dependent and defensive by design — inspect `data/processed/edges_raw.csv` afterward and confirm column mapping before trusting it. See the printed column-mapping TODO at the end of this step.

In [ ]:
!python -m src.data.parse_benchmark --raw_dir data/raw --out_dir data/processed

In [ ]:
import pandas as pd
edges_raw = pd.read_csv('data/processed/edges_raw.csv')
edges_raw.head(20)

**Action needed:** based on the columns printed above, finalize the mapping to the schema expected by `LigandPairDataset`:

`target, ligand_A_id, ligand_B_id, ligand_A_sdf, ligand_B_sdf, protein_pdb, exp_ddg, fep_pred_ddg, source`

Save the cleaned table as `data/processed/edges_final.csv`. This is a one-time manual step since both source repos have their own bespoke file layouts.

## 4. Train

In [ ]:
!python -m src.train --config configs/default.yaml \
    --edges_csv data/processed/edges_final.csv \
    --sdf_dir data/processed/sdf

## 5. Evaluate against experiment AND against FEP+'s own published predictions

In [ ]:
!python -m src.evaluate --checkpoint results/best_model.pt \
    --edges_csv data/processed/edges_final.csv \
    --sdf_dir data/processed/sdf

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

preds = pd.read_csv('results/predictions.csv')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(preds['exp_ddg'], preds['pred_ddg'], alpha=0.6)
axes[0].plot([-5, 5], [-5, 5], 'k--', alpha=0.3)
axes[0].set_xlabel('Experimental ddG (kcal/mol)')
axes[0].set_ylabel('DeltaBind-GNN predicted ddG')
axes[0].set_title('Model vs Experiment')

fep_valid = preds.dropna(subset=['fep_ddg'])
if len(fep_valid) > 0:
    axes[1].scatter(fep_valid['exp_ddg'], fep_valid['fep_ddg'], alpha=0.6, color='orange')
    axes[1].plot([-5, 5], [-5, 5], 'k--', alpha=0.3)
    axes[1].set_xlabel('Experimental ddG (kcal/mol)')
    axes[1].set_ylabel('FEP+ predicted ddG')
    axes[1].set_title('FEP+ vs Experiment (baseline)')

plt.tight_layout()
plt.savefig('results/model_vs_fep_comparison.png', dpi=150)
plt.show()